<a href="https://colab.research.google.com/github/junseok-jay/AI_lab/blob/main/pipeline/bert_split_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 시작

In [19]:
# (필요 시) transformers 설치
# !pip -q install transformers

import os
import torch
import torch.nn as nn
import torch.distributed as dist

from torch.distributed.pipelining import pipeline, SplitPoint
from transformers import BertForSequenceClassification, BertConfig


# -------------------------
# dist init (single process, single GPU)
# -------------------------
def init_dist():
    if dist.is_initialized():
        print("Distributed process group already initialized.")
        return

    os.environ["MASTER_ADDR"] = "127.0.0.1"
    os.environ["MASTER_PORT"] = "29500"
    dist.init_process_group(backend="nccl", rank=0, world_size=1)
    torch.cuda.set_device(0)

def cleanup_dist():
    if dist.is_initialized():
        dist.destroy_process_group()


# -------------------------
# helpers
# -------------------------
def count_params(m: nn.Module) -> int:
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

def ordered_split_candidates_bert(model: nn.Module):
    """
    BertForSequenceClassification 기준:
      - encoder layer 경계들: bert.encoder.layer.{i}
    """
    # get encoder layer count dynamically
    n_layers = len(model.bert.encoder.layer)
    return [f"bert.encoder.layer.{i}" for i in range(n_layers)]

def build_split_spec_bert(model: nn.Module, n_stages: int = 2):
    """
    검증용으로 n_stages=2 기준 param 절반 지점에서 split.
    split_spec은 '해당 모듈 BEGINNING에서 다음 stage 시작'을 의미.
    """
    assert n_stages >= 1
    candidates = ordered_split_candidates_bert(model)

    sizes = [(name, count_params(model.get_submodule(name))) for name in candidates]
    total = sum(p for _, p in sizes)
    target = total / n_stages

    split_names = []
    acc = 0
    for name, p in sizes:
        acc += p
        if len(split_names) < n_stages - 1 and acc >= target:
            split_names.append(name)

    split_spec = {n: SplitPoint.BEGINNING for n in split_names}
    return split_spec, total, sizes


# -------------------------
# wrapper: dict 출력 대신 Tensor(logits)만 반환하도록
# pipeline tracing 시 dict/ModelOutput이 걸림돌이 되는 경우가 있어서 안전하게 래핑
# -------------------------
class BertClsLogitsOnly(nn.Module):
    def __init__(self, hf_model: BertForSequenceClassification):
        super().__init__()
        self.m = hf_model

    def forward(self, input_ids, attention_mask, token_type_ids):
        # return_dict=False로 logits 텐서가 첫번째로 오게
        out = self.m(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            return_dict=False
        )
        logits = out[0]
        return logits


def main():
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU가 필요합니다.")

    # Ensure cleanup is called before init if the main function is re-run within the same kernel session
    cleanup_dist()

    init_dist()
    device = torch.device("cuda:0")

    # 1) BERT 모델 생성 (다운로드 없이 config로 생성)
    #    - 다운로드를 원하면: BertForSequenceClassification.from_pretrained("bert-base-uncased")
    hf_model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2).eval()
    cfg = hf_model.config # Get config from the loaded model

    # 파이프라인 안정성을 위해 wrapper 사용
    model = BertClsLogitsOnly(hf_model).eval()

    # 2) split_spec 생성 (검증용 2-stage 기준)
    #    단일 GPU라도 "파이프 생성만" 보려면 stages를 2로 둬도 됩니다.
    #    split_spec은 hf_model (내부 모델) 기준으로 생성
    split_spec_unwrapped, total_params, _ = build_split_spec_bert(hf_model, n_stages=2)
    print("total params:", f"{total_params:,}")
    print("split_spec (unwrapped):", split_spec_unwrapped)

    # 3) wrapper 모델에 맞게 split_spec 키 업데이트
    split_spec_wrapped = {f"m.{k}": v for k, v in split_spec_unwrapped.items()}
    print("split_spec (wrapped):"), split_spec_wrapped

    # 4) pipeline tracing 용 micro-batch 예시 입력
    #    BERT는 정적 shape가 중요합니다.
    micro_bsz = 2
    seq_len = 128
    input_ids = torch.randint(0, cfg.vocab_size, (micro_bsz, seq_len), dtype=torch.long)
    attention_mask = torch.ones((micro_bsz, seq_len), dtype=torch.long)
    token_type_ids = torch.zeros((micro_bsz, seq_len), dtype=torch.long)

    # 5) pipeline 생성
    #    module은 wrapper (model), split_spec은 wrapper에 맞춰 조정된 것을 사용
    pipe = pipeline(
        module=model,
        mb_args=(input_ids, attention_mask, token_type_ids),
        split_spec=split_spec_wrapped,
    )
    print("✅ pipeline() 생성 성공")
    print("pipe.num_stages =", pipe.num_stages)

    # 6) stage build (GPU 1개이므로 stage_idx=0만 확인)
    stage0 = pipe.build_stage(
        stage_index=0, # Changed from stage_idx to stage_index
        device=device,
        group=dist.group.WORLD,
    )
    print("✅ build_stage(stage_idx=0) 성공")
    print("stage type:", type(stage0))

    cleanup_dist()


if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


total params: 85,054,464
split_spec (unwrapped): {'bert.encoder.layer.5': <SplitPoint.BEGINNING: 1>}
split_spec (wrapped):
✅ pipeline() 생성 성공
pipe.num_stages = 2
✅ build_stage(stage_idx=0) 성공
stage type: <class 'torch.distributed.pipelining.stage._PipelineStage'>


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
